# 🚀 NairaLLM V1.5 — 105K Semantic Pretraining Production Run

**Objective**: Execute the official 105K semantic pretraining foundation training run on Google Colab (Tesla T4 Free GPU).

### Verification Highlights:
- **Architecture**: `NairaTransformer` (128 hidden dim, 4 layers, 4 heads, 512 SwiGLU FFN, 1,242,880 parameters)
- **Dataset**: Dataset A (`semantic_pretrain_v1_5_final.jsonl` — 337 records, 105,141 raw BPE tokens, 105,478 packed tokens)
- **Dataset SHA-256**: `015b4655bde092005b31195025e96df6e80702e7975f05ebf0c6072c1b29ff8f`
- **Hardware**: Google Colab Free Tier (Tesla T4 GPU, ~14.56 GB VRAM, FP16 AMP)
- **Cost**: **$0.00** (Zero paid compute units / strictly free tier policy)
- **Persistent Checkpoints**: Google Drive (`/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretraining/`)
- **Scope**: Complete Pretraining Run + Multi-Domain Semantic Benchmark Evaluation + Final Report + Phase Gate Stop.

## 🛠️ Step 1: Environment & Free GPU Hardware Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"AMP Supported:   True")
else:
    print("WARNING: GPU runtime not active. Please select Runtime -> Change runtime type -> T4 GPU.")

## 📂 Step 2: Mount Google Drive for Persistent Checkpoints

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
colab_ckpt_dir = '/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretraining'
os.makedirs(colab_ckpt_dir, exist_ok=True)
print(f"Persistent Google Drive Checkpoint Directory Ready: {colab_ckpt_dir}")

## 📦 Step 3: Setup NairaLLM Workspace & Dependencies

In [ ]:
# If running directly from cloned repo:
# %cd /content/naira-os

!pip install -q tokenizers psutil

import sys
from pathlib import Path
workspace_root = Path('.').resolve()
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace initialized successfully.")

## ⚡ Step 4: Launch Official 105K Semantic Pretraining Run

In [ ]:
from NairaLLM.training.scripts.run_105k_semantic_pretraining import launch_105k_semantic_pretraining

# Launch 105K semantic pretraining on Tesla T4 GPU (FP16 AMP, CosineAnnealingLR, 90/10 split)
results = launch_105k_semantic_pretraining(
    epochs=25,
    micro_batch_size=4,
    grad_accum_steps=4,
    learning_rate=4e-4,
    min_learning_rate=1e-5,
    max_seq_len=256,
    custom_checkpoint_dir=colab_ckpt_dir,
)

print(f"\n[STATUS] Training Run Status: {results.get('status')}")

## 📊 Step 5: Display Final Evaluation Report & Persistent Checkpoint Artifacts

In [ ]:
from IPython.display import display, Markdown

report_path = 'NairaLLM/evaluation/results/semantic_pretraining_final_report.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content))

print("\nPersistent Checkpoints in Google Drive:")
!ls -lh "/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretraining/"